# Xinyi — independent end-to-end integration prototype

**Purpose:** Unblock the Ensemble task while Ben's agreed preprocessing and Zyen's completed tuning are pending. This is a **separate, provisional experiment**. Do not overwrite team notebooks, treat results as official, or modify `main`.

This workflow:
- Reads local `data/raw/bank-full.csv` and makes an 80/20 stratified split (seed 42)
- Excludes `duration` because it is not available before a marketing call ends; this assumes prediction is required *before* the call
- Treats `unknown` as a category and keeps `pdays=-1` as an informative sentinel
- Encodes categorical features, scales numeric features **inside training-fold pipelines**
- Uses reduced GridSearchCV grids to tune Logistic Regression, Decision Tree, and Random Forest (independent pilot, **not** Zyen's full search)
- Selects two models using training-only cross-validation; selects Voting vs Stacking using training CV
- Evaluates the three base models and **one** chosen ensemble on an untouched test set
- Writes local CSV tables, notes, confusion matrices, comparison chart and ROC curves.

**Important:** Results will differ if Ben chooses a different split, preprocessing, prediction timing or Zyen uses different hyperparameters.

## 1. Check local setup and load the dataset

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src" / "independent_workflow_xinyi.py").exists():
    root = root.parent
assert (root / "src" / "independent_workflow_xinyi.py").exists(), "Open from the project or notebooks folder"
sys.path.insert(0, str(root))

from src.independent_workflow_xinyi import load_split, run

data_file = root / "data/raw/bank-full.csv"
assert data_file.exists(), "Place bank-full.csv in data/raw, and do not push it to GitHub"
X_train, X_test, y_train, y_test = load_split(data_file)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Features:", len(X_train.columns), "(duration excluded)")
print("Positive-class rate in training:", round(y_train.mean(), 4))

## 2. Run the independent experiment

The full dataset can require noticeable CPU time, especially for Random Forest and Stacking. Run it **only once** when you are ready.

By default this cell does not run, so clicking "Run All" is safe. Switch `RUN_FULL_PROTOTYPE` to `True` to launch training. Generated outputs stay under `data/raw/xinyi_local_prototype/`, which is already ignored by your project's Git configuration.

For a quick demonstration only, use `sample_size=1200` — never report that as a full-data result.

In [ ]:
RUN_FULL_PROTOTYPE = False
results_directory = root / "data/raw/xinyi_local_prototype"

if RUN_FULL_PROTOTYPE:
    training_cv, test_results, experiment_info = run(
        data_file,
        results_directory,
        sample_size=None,  # full 45,211 records
    )
else:
    print("Training skipped (safe default). Set RUN_FULL_PROTOTYPE=True to run.")

## 3. Display saved results (only after executing the real pipeline)

In [ ]:
import pandas as pd
from IPython.display import display, Image

if (results_directory / "test_comparison.csv").exists():
    display(pd.read_csv(results_directory / "training_cv.csv"))
    display(pd.read_csv(results_directory / "test_comparison.csv"))
    for filename in ["model_comparison.png", "confusion_matrices.png", "roc_curves.png"]:
        if (results_directory / filename).exists():
            print(filename)
            display(Image(filename=str(results_directory / filename)))
else:
    print("No local results yet. Run the previous cell before displaying figures.")

## 4. Team integration before final report

**Do not present prototype values as the official CW2 results.** Ask Ben to verify the prediction-time assumption, shared preprocessing, and split. Ask Zyen to confirm tuned parameters and saved/cloneable estimators. Then replace the temporary workflow with the agreed models and run final evaluation once.

Your official report should include:
- Training CV scores used to select the two models and explain why the third was excluded
- A justification for Soft Voting or Stacking, including the alternative considered
- Final test metrics for the 3 base models + the selected ensemble
- Confusion matrix interpretation, ROC curves, potential overfitting, efficiency, interpretability
- An evidence-based deployment recommendation **even if a base model outperforms the ensemble**
- Details of any AI assistance and its role, per assignment requirements.

Your existing official working notebook remains `notebooks/03_ensemble_xinyi.ipynb`. This new notebook is a **temporary integration experiment** only.